# 03 — Parquet, Iceberg & Delta Lake

In [1]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 07:05:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
W20260408 07:05:50.043588   767 MemoryArbitrator.cpp:84] Query memory capacity[288.00MB] is set for NOOP arbitrator which has no capacity enforcement
26/04/08 07:05:50 WARN SparkShimProvider: Spark runtime version 4.0.2 is not matched with Gluten's fully tested version 4.0.1


Spark 4.0.2  |  Mode: Gluten/Velox


26/04/08 07:05:53 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=0], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#0L, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 0:>                                                          (0 + 4) / 4]

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## Parquet + Partition Pruning

In [2]:
schema = StructType([
    StructField('id',IntegerType()), StructField('name',StringType()),
    StructField('department',StringType()), StructField('salary',DoubleType()),
    StructField('country',StringType()),
])
employees = spark.createDataFrame([
    (1,'Alice','Engineering',95000.0,'US'),(2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),(4,'Dave','HR',65000.0,'DE'),
], schema)
employees.write.mode('overwrite').partitionBy('country').parquet(f'{DATA_DIR}/employees_parquet')

t0 = time.time()
us_only = spark.read.parquet(f'{DATA_DIR}/employees_parquet').filter(F.col('country')=='US')
print('US rows:', us_only.count(), f'  {time.time()-t0:.2f}s')
us_only.explain()


26/04/08 07:05:58 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=2], due to: [FallbackByBackendSettings] Validation failed on node Exchange


US rows: 2   1.14s
== Physical Plan ==
VeloxColumnarToRow
+- ^(1) FileFileSourceScanExecTransformer parquet [id#11,name#12,department#13,salary#14,country#15] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/employees_parquet], PartitionFilters: [isnotnull(country#15), (country#15 = US)], PushedFilters: [], ReadSchema: struct<id:int,name:string,department:string,salary:double> NativeFilters: []




## Apache Iceberg — ACID · Time Travel · Schema Evolution

In [3]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.demo')
employees.writeTo('local.demo.employees_ice').createOrReplace()
spark.sql("UPDATE local.demo.employees_ice SET salary = salary * 1.1 WHERE department='Engineering'")
spark.sql('SELECT * FROM local.demo.employees_ice').show()


26/04/08 07:05:59 WARN GlutenFallbackReporter: Validation failed for plan: OverwriteByExpression[QueryId=5], due to: [FallbackByBackendSettings] Validation failed on node OverwriteByExpression
26/04/08 07:06:01 WARN GlutenFallbackReporter: Validation failed for plan: ReplaceData[QueryId=6], due to: [FallbackByBackendSettings] Validation failed on node ReplaceData
26/04/08 07:06:01 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice[QueryId=6], due to: 
 - Unsupported scan IcebergCopyOnWriteScan(table=local.demo.employees_ice, type=struct<1: id: optional int, 2: name: optional string, 3: department: optional string, 4: salary: optional double, 5: country: optional string, 2147483646: _file: required string (Path of the file in which a row is stored), 2147483645: _pos: required long (Ordinal position of a row in the source data file)>, filters=[ref(name="department") == "Engineering"], caseSensitive=false)
26/04/08 07:06:01 WARN GlutenFallbackRepor

+---+-----+-----------+------------------+-------+
| id| name| department|            salary|country|
+---+-----+-----------+------------------+-------+
|  3|Carol|Engineering| 96800.00000000001|     US|
|  1|Alice|Engineering|104500.00000000001|     US|
|  2|  Bob|  Marketing|           72000.0|     UK|
|  4| Dave|         HR|           65000.0|     DE|
+---+-----+-----------+------------------+-------+



In [4]:
# Time travel
history = spark.sql('SELECT snapshot_id, committed_at, operation FROM local.demo.employees_ice.snapshots ORDER BY committed_at')
history.show(truncate=False)
first_snap = history.first()['snapshot_id']
spark.read.option('snapshot-id', first_snap).table('local.demo.employees_ice').show()


26/04/08 07:06:04 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice.snapshots[QueryId=8], due to: 
 - Unsupported scan IcebergScan(table=local.demo.employees_ice.snapshots, branch=null, type=struct<1: committed_at: required timestamptz, 2: snapshot_id: required long, 4: operation: optional string>, filters=[], runtimeFilters=[], caseSensitive=false)
26/04/08 07:06:04 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=8], due to: [FallbackByBackendSettings] Validation failed on node TakeOrderedAndProject


+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|966472338712014173 |2026-04-08 06:21:25.817|overwrite|
|6176064989868175206|2026-04-08 06:21:28.074|overwrite|
|1103042709830402275|2026-04-08 06:23:26.21 |overwrite|
|407154164492982870 |2026-04-08 07:06:00.76 |overwrite|
|3868877936129342471|2026-04-08 07:06:02.918|overwrite|
+-------------------+-----------------------+---------+



26/04/08 07:06:04 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice.snapshots[QueryId=9], due to: 
 - Unsupported scan IcebergScan(table=local.demo.employees_ice.snapshots, branch=null, type=struct<1: committed_at: required timestamptz, 2: snapshot_id: required long, 4: operation: optional string>, filters=[], runtimeFilters=[], caseSensitive=false)
26/04/08 07:06:04 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=9], due to: [FallbackByBackendSettings] Validation failed on node TakeOrderedAndProject
26/04/08 07:06:04 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice[QueryId=10], due to: 
 - Unsupported scan IcebergScan(table=local.demo.employees_ice, branch=null, type=struct<1: id: optional int, 2: name: optional string, 3: department: optional string, 4: salary: optional double, 5: country: optional string>, filters=[], runtimeFilters=[], caseSensitive=false)
26/

+---+-----+-----------+-------+-------+
| id| name| department| salary|country|
+---+-----+-----------+-------+-------+
|  1|Alice|Engineering|95000.0|     US|
|  2|  Bob|  Marketing|72000.0|     UK|
|  3|Carol|Engineering|88000.0|     US|
|  4| Dave|         HR|65000.0|     DE|
+---+-----+-----------+-------+-------+



In [5]:
# Schema evolution
spark.sql("ALTER TABLE local.demo.employees_ice ADD COLUMN bonus DOUBLE")
spark.sql("UPDATE local.demo.employees_ice SET bonus = salary * 0.1")
spark.sql('SELECT name, salary, bonus FROM local.demo.employees_ice').show()


26/04/08 07:06:05 WARN GlutenFallbackReporter: Validation failed for plan: ReplaceData[QueryId=12], due to: [FallbackByBackendSettings] Validation failed on node ReplaceData
26/04/08 07:06:05 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice[QueryId=12], due to: 
 - Unsupported scan IcebergCopyOnWriteScan(table=local.demo.employees_ice, type=struct<1: id: optional int, 2: name: optional string, 3: department: optional string, 4: salary: optional double, 5: country: optional string, 7: bonus: optional double, 2147483646: _file: required string (Path of the file in which a row is stored), 2147483645: _pos: required long (Ordinal position of a row in the source data file)>, filters=[true], caseSensitive=false)
26/04/08 07:06:05 WARN GlutenFallbackReporter: Validation failed for plan: ColumnarToRow[QueryId=12], due to: 
 - Unsupported scan IcebergCopyOnWriteScan(table=local.demo.employees_ice, type=struct<1: id: optional int, 2: name: optional stri

+-----+------------------+------------------+
| name|            salary|             bonus|
+-----+------------------+------------------+
|  Bob|           72000.0|            7200.0|
|Carol| 96800.00000000001| 9680.000000000002|
|Alice|104500.00000000001|10450.000000000002|
| Dave|           65000.0|            6500.0|
+-----+------------------+------------------+



26/04/08 07:06:06 WARN GlutenFallbackReporter: Validation failed for plan: BatchScan local.demo.employees_ice[QueryId=13], due to: 
 - Unsupported scan IcebergScan(table=local.demo.employees_ice, branch=null, type=struct<2: name: optional string, 4: salary: optional double, 7: bonus: optional double>, filters=[], runtimeFilters=[], caseSensitive=false)
26/04/08 07:06:06 WARN GlutenFallbackReporter: Validation failed for plan: ColumnarToRow[QueryId=13], due to: 
 - Unsupported scan IcebergScan(table=local.demo.employees_ice, branch=null, type=struct<2: name: optional string, 4: salary: optional double, 7: bonus: optional double>, filters=[], runtimeFilters=[], caseSensitive=false)
26/04/08 07:06:06 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=13], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(name#333, Some(Etc/UTC)), class name: ToPrettySt

## Delta Lake — ACID · Time Travel · OPTIMIZE

In [6]:
employees.write.format('delta').mode('overwrite').partitionBy('country').save(f'{DATA_DIR}/employees_delta')
spark.sql(f"CREATE TABLE IF NOT EXISTS employees_delta USING delta LOCATION '{DATA_DIR}/employees_delta'")
spark.sql("UPDATE employees_delta SET salary = salary * 1.2 WHERE country='US'")
spark.sql('SELECT * FROM employees_delta').show()


26/04/08 07:06:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/08 07:06:07 WARN GlutenFallbackReporter: Validation failed for plan: Scan json [QueryId=15], due to: 
 - Unsupported file format UnknownFormat.
26/04/08 07:06:07 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=15], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: UDF name is not found!
26/04/08 07:06:07 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=15], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:06:09 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=16], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:06:14 WARN GlutenFallbackReporter: Validation failed for plan: Scan json [QueryId=21], due to: 
 - Unsupported 

+---+-----+-----------+--------+-------+
| id| name| department|  salary|country|
+---+-----+-----------+--------+-------+
|  3|Carol|Engineering|105600.0|     US|
|  1|Alice|Engineering|114000.0|     US|
|  2|  Bob|  Marketing| 72000.0|     UK|
|  4| Dave|         HR| 65000.0|     DE|
+---+-----+-----------+--------+-------+



In [7]:
# Time travel
spark.read.format('delta').option('versionAsOf', 0).load(f'{DATA_DIR}/employees_delta').show()
spark.sql('DESCRIBE HISTORY employees_delta').select('version','timestamp','operation').show()


26/04/08 07:06:20 WARN GlutenFallbackReporter: Validation failed for plan: Scan json [QueryId=32], due to: 
 - Unsupported file format UnknownFormat.
26/04/08 07:06:20 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=32], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: UDF name is not found!
26/04/08 07:06:20 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=32], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:06:20 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=35], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:06:21 WARN GlutenFallbackReporter: Validation failed for plan: Scan parquet [QueryId=31], due to: 
 - Unsupported file format UnknownFormat.
26/04/08 07:06:21 WARN GlutenFallbackReporter: Validation failed for plan: ColumnarToRow[QueryId=31], due to: 
 - Unsupported file format UnknownFormat.
26/04/

+---+-----+-----------+-------+-------+
| id| name| department| salary|country|
+---+-----+-----------+-------+-------+
|  3|Carol|Engineering|88000.0|     US|
|  1|Alice|Engineering|95000.0|     US|
|  2|  Bob|  Marketing|72000.0|     UK|
|  4| Dave|         HR|65000.0|     DE|
+---+-----+-----------+-------+-------+

+-------+--------------------+---------+
|version|           timestamp|operation|
+-------+--------------------+---------+
|      3|2026-04-08 07:06:...|   UPDATE|
|      2|2026-04-08 07:06:...|    WRITE|
|      1|2026-04-08 06:23:...|   UPDATE|
|      0|2026-04-08 06:23:...|    WRITE|
+-------+--------------------+---------+



26/04/08 07:06:22 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=39], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(version#1927L, Some(Etc/UTC)), class name: ToPrettyString.
